In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_absolute_error, mean_squared_error, classification_report

In [2]:
data_path = "../data/processed/data_processed.csv"
df = pd.read_csv(data_path, parse_dates=['date'])

In [3]:
df_sort = df.sort_values(by=['date', 'city']).reset_index(drop=True)
df_sort.head(24)

,city,latitude,longitude,date,temperature_2m_mean,rain_sum,precipitation_hours,weather_code,wind_speed_10m_mean,relative_humidity_2m_mean,...,dewpoint_2m_mean_lag_2,surface_pressure_mean_lag_2,cloudcover_mean_lag_2,wind_speed_10m_mean_lag_2,temperature_2m_mean_rolling_mean_3,relative_humidity_2m_mean_rolling_mean_3,dewpoint_2m_mean_rolling_mean_3,surface_pressure_mean_rolling_mean_3,cloudcover_mean_rolling_mean_3,wind_speed_10m_mean_rolling_mean_3
0,0,10.5417,107.2429,2014-01-08,-0.351351,0.125,1.0,1,0.510638,-0.773585,...,-0.993846,-0.197183,-0.412916,1.042553,-0.441441,-0.918239,-0.943932,-0.144422,-0.002712,6.950355e-01
1,1,16.0544,108.2022,2014-01-08,-1.486486,0.000,0.0,0,-0.212766,0.509434,...,-1.526154,0.820926,-1.465548,0.425532,-1.549550,0.246541,-1.275214,0.840599,-1.167218,2.198582e-01
2,2,10.9465,106.8340,2014-01-08,-0.162162,0.000,0.0,0,-0.617021,-1.007547,...,-0.899487,0.256204,-0.608187,-0.361702,-0.189189,-1.044182,-0.884103,0.316790,-0.261378,-3.971631e-01
3,3,21.0278,105.8342,2014-01-08,-2.324324,0.750,5.0,1,0.340426,-0.332075,...,-1.757949,0.393696,-0.325451,0.659574,-1.978604,-0.030189,-1.811282,0.622625,-0.100347,2.553191e-01
4,4,10.7769,106.7009,2014-01-08,-0.081081,0.000,0.0,0,-0.489362,-1.132075,...,-0.925128,0.156271,-0.533944,-0.276596,-0.162162,-1.073270,-0.923761,0.213280,-0.178998,-2.588652e-01
5,5,16.4637,107.5909,2014-01-08,-1.378378,0.000,0.0,0,-0.531915,0.596226,...,-1.417436,0.684105,-1.774727,0.659574,-1.342342,0.167296,-1.130598,0.736195,-1.343165,7.092199e-02
6,6,12.2585,109.0526,2014-01-08,-1.027027,0.000,0.0,0,1.212766,-0.033962,...,-1.368205,0.594232,-1.160437,1.255319,-1.027027,-0.402516,-1.155556,0.596691,-0.720061,1.127660e+00
7,7,10.5336,106.4110,2014-01-08,-0.243243,0.000,0.0,0,-0.489362,-0.728302,...,-0.779487,0.305835,-0.364099,-0.255319,-0.288288,-0.838994,-0.789060,0.358596,-0.087465,-2.836879e-01
8,8,15.5394,108.0191,2014-01-08,-1.432432,0.000,0.0,0,-0.702128,0.384906,...,-1.510769,0.179745,-1.433003,-0.297872,-1.423423,0.084277,-1.268718,0.197407,-0.971608,-4.680851e-01
9,9,21.0064,107.2925,2014-01-08,-2.540541,0.700,2.0,1,1.664894,-0.177358,...,-1.758974,0.799463,-0.101704,0.297872,-2.333333,0.394969,-1.891966,0.963783,-0.046784,6.187943e-01


In [4]:
def split_data(df, val_year = 2, test_year = 1):
    years = df['date'].dt.year.unique()
    print(years)
    val_years = years[-(val_year+test_year):-test_year]
    print(val_years)
    test_years = years[-test_year:]
    train_data = df[~df['date'].dt.year.isin(val_years)].reset_index(drop=True)
    val_data = df[df['date'].dt.year.isin(val_years)].reset_index(drop=True)
    test_data = df[df['date'].dt.year.isin(test_years)].reset_index(drop=True)
    return train_data, val_data, test_data

In [5]:
train_data, val_data, test_data = split_data(df_sort, val_year=2, test_year=1)

[2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024]
[2022 2023]


In [6]:
train_data.to_csv("../data/processed/train_data.csv", index=False)
val_data.to_csv("../data/processed/val_data.csv", index=False)
test_data.to_csv("../data/processed/test_data.csv", index=False)

In [7]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39372 entries, 0 to 39371
Data columns (total 58 columns):
 #   Column                                    Non-Null Count  Dtype         
---  ------                                    --------------  -----         
 0   city                                      39372 non-null  int64         
 1   latitude                                  39372 non-null  float64       
 2   longitude                                 39372 non-null  float64       
 3   date                                      39372 non-null  datetime64[ns]
 4   temperature_2m_mean                       39372 non-null  float64       
 5   rain_sum                                  39372 non-null  float64       
 6   precipitation_hours                       39372 non-null  float64       
 7   weather_code                              39372 non-null  int64         
 8   wind_speed_10m_mean                       39372 non-null  float64       
 9   relative_humidity_2m_mean   

In [8]:
val_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 58 columns):
 #   Column                                    Non-Null Count  Dtype         
---  ------                                    --------------  -----         
 0   city                                      8760 non-null   int64         
 1   latitude                                  8760 non-null   float64       
 2   longitude                                 8760 non-null   float64       
 3   date                                      8760 non-null   datetime64[ns]
 4   temperature_2m_mean                       8760 non-null   float64       
 5   rain_sum                                  8760 non-null   float64       
 6   precipitation_hours                       8760 non-null   float64       
 7   weather_code                              8760 non-null   int64         
 8   wind_speed_10m_mean                       8760 non-null   float64       
 9   relative_humidity_2m_mean     

In [9]:
test_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4392 entries, 0 to 4391
Data columns (total 58 columns):
 #   Column                                    Non-Null Count  Dtype         
---  ------                                    --------------  -----         
 0   city                                      4392 non-null   int64         
 1   latitude                                  4392 non-null   float64       
 2   longitude                                 4392 non-null   float64       
 3   date                                      4392 non-null   datetime64[ns]
 4   temperature_2m_mean                       4392 non-null   float64       
 5   rain_sum                                  4392 non-null   float64       
 6   precipitation_hours                       4392 non-null   float64       
 7   weather_code                              4392 non-null   int64         
 8   wind_speed_10m_mean                       4392 non-null   float64       
 9   relative_humidity_2m_mean     

### CONFIG

In [10]:
# Col to predict
target_column = 'rain_sum'
side_target = ['precipitation_hours', 'weather_code', "date"]

drop_cols = ['surface_pressure_mean', 'wind_speed_10m_mean','cloudcover_mean', 'temperature_2m_mean', 'relative_humidity_2m_mean', 'rain_sum','precipitation_hours', 'weather_code', 'date']

# Define training and testing sets
X_train = train_data.drop(columns=drop_cols)

X_test = val_data.drop(columns=drop_cols)



# METRICS


# XGBOOST

1. Model

In [37]:
import xgboost as xgb
import joblib

rf_params = {'n_estimators':1000,
    'learning_rate':0.05,
    'max_depth':6,
    'objective':'reg:squarederror',
    'enable_categorical':True,
    'device':'cuda' if torch.cuda.is_available() else 'cpu'}

models = {
    # --- NHÓM 1: DRIVERS (Dự báo trước) ---
    'surface_pressure_mean': xgb.XGBRegressor(**rf_params),
    'wind_speed_10m_mean': xgb.XGBRegressor(**rf_params),
    'cloudcover_mean': xgb.XGBRegressor(**rf_params),
    'temperature_2m_mean': xgb.XGBRegressor(**rf_params),
    
    # --- NHÓM 2: MOISTURE (Dự báo sau Drivers) ---
    'relative_humidity_2m_mean': xgb.XGBRegressor(**rf_params),
    
    # --- NHÓM 3: TARGETS (Dự báo cuối cùng) ---
    'rain_sum': xgb.XGBRegressor(**rf_params), 
    'precipitation_hours': xgb.XGBRegressor(**rf_params),
    'weather_code': xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    objective='multi:softmax',
    num_class=len(train_data['weather_code'].unique()),
    enable_categorical=True,
    device='cuda' if torch.cuda.is_available() else 'cpu'
) 
}

evaluation_results = {}
train_features = {}
for name, model in models.items():
    X_test_features = X_test.copy()
    X_train_features = X_train.copy()
    if len(train_features) > 0:
        for target, feature in train_features.items():
            X_train_features = pd.concat([X_train_features, train_data[target]], axis=1)
            X_test_features = pd.concat([X_test_features, pd.Series(feature, name=target)], axis=1)
    model.fit(X_train_features, train_data[name])
    Y_pred = model.predict(X_test_features)
    train_features[name] = Y_pred
    y_true = val_data[name]
    if name == 'weather_code':
        report = classification_report(y_true, Y_pred, output_dict=True)
        evaluation_results[name] = report
        print(f"Weather Code - Classification Report:\n{classification_report(y_true, Y_pred)}")
    else:
        mae = mean_absolute_error(y_true=y_true, y_pred=Y_pred)
        mse = mean_squared_error(y_true=y_true, y_pred=Y_pred)
        evaluation_results[name] = {'MAE': mae, 'MSE': mse, 'RMSE': np.sqrt(mse)}
        print(f"{name} - MAE: {mae}, MSE: {mse}, RMSE: {np.sqrt(mse)}")
    joblib.dump(model, f"../models/xgboost/{name}_model.pkl")

surface_pressure_mean - MAE: 0.024104521929764203, MSE: 0.002629829236041741, RMSE: 0.05128186069207845
wind_speed_10m_mean - MAE: 0.021541308372019195, MSE: 0.004523644700752878, RMSE: 0.06725804562097294
cloudcover_mean - MAE: 0.016903028270180274, MSE: 0.0006808958489645818, RMSE: 0.026093981086920827
temperature_2m_mean - MAE: 0.03470276236147016, MSE: 0.003487008973821008, RMSE: 0.059050901549603865
relative_humidity_2m_mean - MAE: 0.027344054543939036, MSE: 0.002763770238260236, RMSE: 0.05257157252984008
rain_sum - MAE: 0.416994371719012, MSE: 3.926792175432673, RMSE: 1.981613528272522
precipitation_hours - MAE: 0.29066130442706417, MSE: 0.22091320008378343, RMSE: 0.4700140424325463
Weather Code - Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.97      0.97      1971
           1       0.99      0.99      0.99      6789

    accuracy                           0.98      8760
   macro avg       0.98      0.98      0.98   

# RANDOM_FOREST

In [38]:
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

rf_params = {    'n_estimators':500,
    'max_depth':6,
    'min_samples_leaf' : 10,
    'min_samples_split' : 20,
    'max_features' : 'sqrt',
    'n_jobs':-1}

models = {
    # --- NHÓM 1: DRIVERS (Dự báo trước) ---
    'surface_pressure_mean': RandomForestRegressor(**rf_params),
    'wind_speed_10m_mean': RandomForestRegressor(**rf_params),
    'cloudcover_mean': RandomForestRegressor(**rf_params),
    'temperature_2m_mean': RandomForestRegressor(**rf_params),
    
    # --- NHÓM 2: MOISTURE (Dự báo sau Drivers) ---
    'relative_humidity_2m_mean': RandomForestRegressor(**rf_params),
    
    # --- NHÓM 3: TARGETS (Dự báo cuối cùng) ---
    'rain_sum': RandomForestRegressor(**rf_params), 
    'precipitation_hours': RandomForestRegressor(**rf_params),
    'weather_code': RandomForestClassifier(**rf_params, class_weight='balanced') 
}

evaluation_results = {}
train_features = {}
for name, model in models.items():
    X_test_features = X_test.copy()
    X_train_features = X_train.copy()
    if len(train_features) > 0:
        for target, feature in train_features.items():
            X_train_features = pd.concat([X_train_features, train_data[target]], axis=1)
            X_test_features = pd.concat([X_test_features, pd.Series(feature, name=target)], axis=1)
    model.fit(X_train_features, train_data[name])
    Y_pred = model.predict(X_test_features)
    train_features[name] = Y_pred
    y_true = val_data[name]
    if name == 'weather_code':
        report = classification_report(y_true, Y_pred, output_dict=True)
        evaluation_results[name] = report
        print(f"Weather Code - Classification Report:\n{classification_report(y_true, Y_pred)}")
    else:
        mae = mean_absolute_error(y_true=y_true, y_pred=Y_pred)
        mse = mean_squared_error(y_true=y_true, y_pred=Y_pred)
        evaluation_results[name] = {'MAE': mae, 'MSE': mse, 'RMSE': np.sqrt(mse)}
        print(f"{name} - MAE: {mae}, MSE: {mse}, RMSE: {np.sqrt(mse)}")
    joblib.dump(model, f"../models/xgboost/{name}_model.pkl")

surface_pressure_mean - MAE: 0.1778579205390185, MSE: 0.05992957981185857, RMSE: 0.24480518746925803
wind_speed_10m_mean - MAE: 0.2950877972097874, MSE: 0.16227049012516695, RMSE: 0.4028281148643512
cloudcover_mean - MAE: 0.29274615344235744, MSE: 0.1364483137950375, RMSE: 0.36938910892856264
temperature_2m_mean - MAE: 0.14876553197988718, MSE: 0.045155203042797284, RMSE: 0.2124975365570088
relative_humidity_2m_mean - MAE: 0.23564670418824168, MSE: 0.1042668495957226, RMSE: 0.3229037776114157
rain_sum - MAE: 3.289406705303167, MSE: 41.97630185972823, RMSE: 6.478912089211292
precipitation_hours - MAE: 2.7791479293661716, MSE: 13.383347152838585, RMSE: 3.6583257308280497
Weather Code - Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.29      0.45      1971
           1       0.83      1.00      0.91      6789

    accuracy                           0.84      8760
   macro avg       0.91      0.65      0.68      8760
weighted avg

# ROLLING PREDICT

LoadModel

In [13]:
models = {
    'surface_pressure_mean': joblib.load("../models/xgboost/surface_pressure_mean_model.pkl"),
    'wind_speed_10m_mean': joblib.load("../models/xgboost/wind_speed_10m_mean_model.pkl"),
    'cloudcover_mean': joblib.load("../models/xgboost/cloudcover_mean_model.pkl"),
    'temperature_2m_mean': joblib.load("../models/xgboost/temperature_2m_mean_model.pkl"),
    'relative_humidity_2m_mean': joblib.load("../models/xgboost/relative_humidity_2m_mean_model.pkl"),
    'rain_sum': joblib.load("../models/xgboost/rain_sum_model.pkl"),
    'precipitation_hours': joblib.load("../models/xgboost/precipitation_hours_model.pkl"),
    'weather_code': joblib.load("../models/xgboost/weather_code_model.pkl")
}

In [39]:
target_cols = ["rain_sum", "weather_code", "precipitation_hours"]
dynamic_cols = ['temperature_2m_mean', 
    'relative_humidity_2m_mean', 
    'dewpoint_2m_mean', 
    'surface_pressure_mean', 
    'cloudcover_mean', 
    'wind_speed_10m_mean']
predict_cols = ['surface_pressure_mean', 'wind_speed_10m_mean','cloudcover_mean', 'temperature_2m_mean', 'relative_humidity_2m_mean', 'rain_sum','precipitation_hours', 'weather_code']
lags_target = [1,2,3,7]
window_rolling_target = [3,7]
lags_dynamic = [1,2]
window_rolling_dynamic = [3]

In [65]:
def generate_lag_rolling_features(df_):
    input = df_.groupby('city').tail(1).copy()
    next_date = input['date'] + pd.Timedelta(days=1)
    input['date'] = next_date
    
    for col in predict_cols:
        input[col] = np.nan
    
    month_max = 12
    week_day_max = 7
    input['year'] = next_date.dt.year
    input['day_sin'] = np.sin(2 * np.pi * next_date.dt.day / next_date.dt.days_in_month)
    input['day_cos'] = np.cos(2 * np.pi * next_date.dt.day / next_date.dt.days_in_month)  
    input['dayofweek_sin'] = np.sin(2 * np.pi * next_date.dt.dayofweek / week_day_max)
    input['dayofweek_cos'] = np.cos(2 * np.pi * next_date.dt.dayofweek / week_day_max)
    input['month_sin'] = np.sin(2 * np.pi * next_date.dt.month / month_max)
    input['month_cos'] = np.cos(2 * np.pi * next_date.dt.month / month_max)
    input['quarter_sin'] = np.sin(2 * np.pi * next_date.dt.quarter / 4)
    input['quarter_cos'] = np.cos(2 * np.pi * next_date.dt.quarter / 4)
    temp_df = pd.concat([df_, input], ignore_index=True)
    temp_df = temp_df.sort_values(['date', 'city']).reset_index(drop=True)
    for col in target_cols:
        for lag in lags_target:
            temp_df[f"{col}_lag_{lag}"] = temp_df.groupby('city')[col].shift(lag)
        for window in window_rolling_target:
            temp_df[f"{col}_rolling_mean_{window}"] = temp_df.groupby('city')[col].transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
            
    for col in dynamic_cols:
        for lag in lags_dynamic:
            temp_df[f"{col}_lag_{lag}"] = temp_df.groupby('city')[col].shift(lag)
        for window in window_rolling_dynamic:
            temp_df[f"{col}_rolling_mean_{window}"] = temp_df.groupby('city')[col].transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    final_input = temp_df.groupby('city').tail(1)
    final_input = final_input.drop(columns=drop_cols)
    print(final_input.columns)
    return final_input, temp_df

In [66]:
def predict_weather(models, data, rolling_window=7): # hiện tại data nhận vào là train_data
    data_sorted = data.sort_values(by=['date', 'city']).reset_index(drop=True)
    predictions = {}
    data_past = data_sorted.groupby('city').tail(rolling_window+7).reset_index(drop=True)
    for i in range(rolling_window):
        input_row, data_past = generate_lag_rolling_features(data_past)
        predictions[f'day_{i+1}'] = {}
        for target, model in models.items():    
            pred = model.predict(input_row)
            input_row[target] = pred
            predictions[f'day_{i+1}'][target] = pred
            data_past.loc[input_row.index, target] = pred
    return predictions, data_past

In [67]:
class RollingPredictor:
    def __init__(self, models, rolling_window=7):
        self.models = models
        self.rolling_window = rolling_window
        
    def predict(self, data):
        return predict_weather(self.models, data, self.rolling_window)

In [54]:
test_rolling = test_data.groupby('city').head(30).reset_index(drop=True)
test_rolling_predictions, test_rolling_data = predict_weather(models, test_rolling, rolling_window=7)
y_true = test_data.groupby('city').head(37).reset_index(drop=True)

Index(['city', 'latitude', 'longitude', 'dewpoint_2m_mean', 'year',
       'month_sin', 'month_cos', 'day_sin', 'day_cos', 'dayofweek_sin',
       'dayofweek_cos', 'quarter_sin', 'quarter_cos', 'rain_sum_lag_1',
       'weather_code_lag_1', 'precipitation_hours_lag_1', 'rain_sum_lag_2',
       'weather_code_lag_2', 'precipitation_hours_lag_2', 'rain_sum_lag_3',
       'weather_code_lag_3', 'precipitation_hours_lag_3', 'rain_sum_lag_7',
       'weather_code_lag_7', 'precipitation_hours_lag_7',
       'rain_sum_rolling_mean_3', 'weather_code_rolling_mean_3',
       'precipitation_hours_rolling_mean_3', 'rain_sum_rolling_mean_7',
       'weather_code_rolling_mean_7', 'precipitation_hours_rolling_mean_7',
       'temperature_2m_mean_lag_1', 'relative_humidity_2m_mean_lag_1',
       'dewpoint_2m_mean_lag_1', 'surface_pressure_mean_lag_1',
       'cloudcover_mean_lag_1', 'wind_speed_10m_mean_lag_1',
       'temperature_2m_mean_lag_2', 'relative_humidity_2m_mean_lag_2',
       'dewpoint_2m_me

In [55]:
y_true_rain = y_true.groupby('city')['rain_sum'].tail(7).values
y_true_precip_hours = y_true.groupby('city')['precipitation_hours'].tail(7).values
y_true_weather_code = y_true.groupby('city')['weather_code'].tail(7).values

In [68]:
y_true_rain

array([0.   , 0.   , 0.   , 0.2  , 0.   , 0.   , 0.2  , 0.   , 0.   ,
       0.5  , 0.2  , 0.   , 0.   , 0.   , 0.   , 0.2  , 0.   , 0.   ,
       0.1  , 0.   , 0.   , 0.1  , 0.2  , 0.1  , 0.   , 0.2  , 0.1  ,
       0.4  , 0.   , 0.   , 0.   , 0.125, 0.   , 1.1  , 0.6  , 0.6  ,
       0.   , 0.   , 0.   , 0.9  , 0.   , 0.   , 0.   , 0.   , 0.1  ,
       2.05 , 2.6  , 2.4  , 0.   , 0.   , 0.   , 1.   , 0.   , 0.   ,
       0.   , 0.   , 0.   , 0.5  , 1.7  , 0.5  , 0.   , 0.2  , 0.2  ,
       1.2  , 0.125, 0.   , 0.   , 0.1  , 0.2  , 1.6  , 4.925, 1.1  ,
       0.   , 0.1  , 0.   , 3.3  , 0.   , 0.   , 0.   , 0.1  , 0.1  ,
       1.3  , 8.1  , 0.4  ])

In [56]:
y_true.groupby('city').tail(1)

,city,latitude,longitude,date,temperature_2m_mean,rain_sum,precipitation_hours,weather_code,wind_speed_10m_mean,relative_humidity_2m_mean,...,dewpoint_2m_mean_lag_2,surface_pressure_mean_lag_2,cloudcover_mean_lag_2,wind_speed_10m_mean_lag_2,temperature_2m_mean_rolling_mean_3,relative_humidity_2m_mean_rolling_mean_3,dewpoint_2m_mean_rolling_mean_3,surface_pressure_mean_rolling_mean_3,cloudcover_mean_rolling_mean_3,wind_speed_10m_mean_rolling_mean_3
432,0,10.5417,107.2429,2024-02-06,0.027027,0.0,0.0,0,1.510638,-0.422642,...,-0.332308,0.340711,-1.150267,0.914894,0.006757,-0.382390,-0.279316,0.357702,-0.842105,1.120567
433,1,16.0544,108.2022,2024-02-06,-0.513514,0.1,1.0,1,1.000000,0.483019,...,-0.332308,0.899396,-0.830918,0.510638,-0.621622,0.612579,-0.246154,0.951263,-0.702432,0.737589
434,2,10.9465,106.8340,2024-02-06,0.283784,0.0,0.0,0,0.361702,-0.898113,...,-0.381538,0.801476,-1.237732,0.000000,0.202703,-0.739623,-0.343932,0.812207,-0.758708,0.170213
435,3,21.0278,105.8342,2024-02-06,-1.405405,3.3,17.0,1,1.106383,0.437736,...,-0.591795,0.403085,-0.093313,1.170213,-1.018018,0.217610,-0.819145,0.615918,0.301805,1.319149
436,4,10.7769,106.7009,2024-02-06,0.324324,0.0,0.0,0,1.026596,-0.886792,...,-0.376410,0.681422,-1.162471,0.595745,0.261261,-0.767296,-0.308034,0.695730,-0.783117,0.718085
437,5,16.4637,107.5909,2024-02-06,-0.378378,0.0,0.0,1,-0.234043,0.479245,...,-0.230769,0.765929,-0.874650,-0.255319,-0.441441,0.499371,-0.157949,0.792757,-1.212984,-0.141844
438,6,12.2585,109.0526,2024-02-06,-0.378378,0.0,0.0,0,-0.255319,0.237736,...,-0.417436,0.885983,-0.630562,-0.148936,-0.450450,0.270440,-0.311795,0.936061,-0.433257,-0.219858
439,7,10.5336,106.4110,2024-02-06,0.162162,0.1,1.0,1,0.702128,-0.490566,...,-0.365128,0.835010,-0.881770,0.276596,0.126126,-0.494340,-0.267350,0.851330,-0.689211,0.368794
440,8,15.5394,108.0191,2024-02-06,-0.162162,0.1,1.0,1,-0.765957,0.139623,...,-0.163077,0.270959,-0.745487,-0.728723,-0.234234,0.262893,-0.115214,0.325732,-0.464107,-0.718085
441,9,21.0064,107.2925,2024-02-06,-2.162162,1.3,10.0,1,2.489362,0.433962,...,-0.772308,0.854460,0.504450,0.553191,-1.900901,0.693082,-1.331624,1.019897,0.504450,1.828014


In [57]:
test_rolling_data.groupby('city').tail(1)

,city,latitude,longitude,date,temperature_2m_mean,rain_sum,precipitation_hours,weather_code,wind_speed_10m_mean,relative_humidity_2m_mean,...,dewpoint_2m_mean_lag_2,surface_pressure_mean_lag_2,cloudcover_mean_lag_2,wind_speed_10m_mean_lag_2,temperature_2m_mean_rolling_mean_3,relative_humidity_2m_mean_rolling_mean_3,dewpoint_2m_mean_rolling_mean_3,surface_pressure_mean_rolling_mean_3,cloudcover_mean_rolling_mean_3,wind_speed_10m_mean_rolling_mean_3
240,0,10.5417,107.2429,2024-02-06,-0.054368,0.081712,0.486822,0.0,1.443242,-1.296917,...,-1.056410,0.357244,-0.591376,1.531790,-0.052457,-1.322441,-1.056410,0.356450,-0.595634,1.509500
241,1,16.0544,108.2022,2024-02-06,-1.089273,3.428298,12.811632,1.0,0.332287,0.274121,...,-0.993846,1.037704,0.026258,0.302007,-1.097916,0.268842,-0.993846,1.019089,0.018851,0.315374
242,2,10.9465,106.8340,2024-02-06,0.043907,0.082111,0.486979,0.0,0.179914,-1.334658,...,-1.057436,0.531814,-0.558682,0.159377,0.050931,-1.367824,-1.057436,0.519686,-0.568130,0.168089
243,3,21.0278,105.8342,2024-02-06,-2.343219,1.477859,6.164235,1.0,-0.010121,-0.415144,...,-2.579487,1.331199,0.033806,-0.035168,-2.361715,-0.423734,-2.579487,1.329530,0.040996,-0.030552
244,4,10.7769,106.7009,2024-02-06,-0.076510,0.081843,0.480134,0.0,0.244822,-1.351244,...,-1.118974,0.521538,-0.572452,0.253658,-0.044299,-1.419746,-1.118974,0.513977,-0.574064,0.250976
245,5,16.4637,107.5909,2024-02-06,-1.551303,2.250343,9.579362,1.0,0.304628,0.160663,...,-1.508205,1.067294,-0.020613,0.292190,-1.569323,0.133436,-1.508205,1.051699,-0.021922,0.298544
246,6,12.2585,109.0526,2024-02-06,-0.916890,1.348037,6.406853,1.0,0.146629,-0.034507,...,-0.872821,0.831582,-0.242919,0.145415,-0.910463,-0.058342,-0.872821,0.814498,-0.241902,0.145099
247,7,10.5336,106.4110,2024-02-06,-0.009610,0.081491,0.478276,0.0,0.684608,-1.290007,...,-1.001026,0.540839,-0.587286,0.686688,-0.014112,-1.298670,-1.001026,0.527252,-0.591139,0.687263
248,8,15.5394,108.0191,2024-02-06,-1.013095,3.995705,13.731944,1.0,-0.202126,0.286119,...,-0.907692,0.774941,0.058038,-0.234611,-1.018248,0.284503,-0.907692,0.769923,0.053160,-0.227276
249,9,21.0064,107.2925,2024-02-06,-1.945443,2.830204,10.770983,1.0,0.287608,0.108838,...,-1.988718,1.178716,0.130835,0.163319,-1.965933,0.117524,-1.988718,1.165021,0.123589,0.200623


In [58]:
y_pred_rain = []
for target, val in test_rolling_predictions.items():
    y_pred_rain.extend(val['rain_sum'])
y_pred_rain = np.array(y_pred_rain)

In [59]:
y_true_rain.shape, y_pred_rain.shape

((84,), (84,))

In [60]:
y_pred_precip_hours = []
for target, val in test_rolling_predictions.items():
    y_pred_precip_hours.extend(val['precipitation_hours'])
y_pred_precip_hours = np.array(y_pred_precip_hours)

In [61]:
y_pred_weather_code = []
for target, val in test_rolling_predictions.items():
    y_pred_weather_code.extend(val['weather_code'])
y_pred_weather_code = np.array(y_pred_weather_code)

In [62]:
mae = mean_absolute_error(y_true=y_true_rain, y_pred=y_pred_rain)
mse = mean_squared_error(y_true=y_true_rain, y_pred=y_pred_rain)
print(f"Rain Sum - MAE: {mae}, MSE: {mse}, RMSE: {np.sqrt(mse)}")

Rain Sum - MAE: 1.2261450105124054, MSE: 3.4164562392999436, RMSE: 1.8483658294017296


In [63]:
mae = mean_absolute_error(y_true=y_true_precip_hours, y_pred=y_pred_precip_hours)
mse = mean_squared_error(y_true=y_true_precip_hours, y_pred=y_pred_precip_hours)
print(f"Precip Hours - MAE: {mae}, MSE: {mse}, RMSE: {np.sqrt(mse)}")

Precip Hours - MAE: 4.240393460145373, MSE: 37.58518569702773, RMSE: 6.130675794480387


In [64]:
rp = classification_report(y_true_weather_code, y_pred_weather_code)
print(rp)

              precision    recall  f1-score   support

           0       0.64      0.47      0.55        38
           1       0.64      0.78      0.71        46

    accuracy                           0.64        84
   macro avg       0.64      0.63      0.63        84
weighted avg       0.64      0.64      0.63        84

